# Text-driven object removal and replacement — demo

Removes or replaces an object in a photo from one plain-English instruction, **on CPU only**.

```
instruction → Parse (regex rules, Qwen2.5-0.5B Q4 if unusual) → Ground (YOLOv8n-seg, YOLO-World fallback)
            → Segment → Remove (MI-GAN) → Insert (SD-1.5 inpaint + LCM-LoRA + TAESD, or a stored cutout)
```

Expected on a 2-vCPU Colab CPU runtime: first run ≈ 10 min (installs + 4.7 GB of weights);
removal ≈ 10 s with the LLM parser (≈ 2 s without); text-defined replacement ≈ 90 s; fast replacement ≈ 3 s.
Measurements and design rationale: `docs/benchmark_report.md`, `docs/approach.md`.

## 1. Setup

Run this notebook from the repository root (or its `notebooks/` folder). On Colab, first put the
repository at `/content/task2-text-object-editing` (clone the private repository or upload it).

In [ ]:
import os
import sys
from pathlib import Path

for candidate in [Path.cwd(), Path.cwd().parent, Path("/content/task2-text-object-editing")]:
    if (candidate / "src" / "editing").is_dir():
        os.chdir(candidate)
        break
else:
    raise SystemExit("Repository not found: open the notebook inside the repository or place it in /content.")

sys.path.insert(0, str(Path.cwd() / "src"))
IN_COLAB = "google.colab" in sys.modules
print("repository:", Path.cwd(), "| colab:", IN_COLAB)

In [ ]:
# Dependencies (skip if already installed). If Colab asks to restart after this cell,
# restart the runtime and continue from the next cell.
!pip install -q -r requirements.txt -r requirements-diffusion.txt
!pip install -q -r requirements-llm.txt || echo "llama-cpp-python wheel unavailable: the parser will use the transformers backend"

In [ ]:
# Weights for every profile (~4.7 GB, cached in weights/ and the Hugging Face cache).
# Use --profile core for removal of COCO objects only (~34 MB).
!python scripts/download_weights.py --profile full

## 2. Load the editor

In [ ]:
import json
import sys
from pathlib import Path

import cv2
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd() / "src"))
from editing import Editor, ObjectNotFoundError, load_config
from editing.utils import read_image, system_info, write_image

editor = Editor(load_config("configs/default.yaml"))
print(json.dumps(system_info(editor.threads), indent=2))


def run(image_name: str, instruction: str, mode: str | None = None):
    """Edit one sample image and show original | removal mask | result."""
    image = read_image(f"data/samples/{image_name}")
    try:
        result = editor.edit(image, instruction, mode=mode)
    except (ObjectNotFoundError, ValueError) as exc:
        print(f"{instruction!r}: {exc}")
        return None
    out_path = write_image(f"outputs/demo/{Path(image_name).stem}_{abs(hash(instruction)) % 10**6}.jpg", result.image)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, img, title in zip(axes, [image, result.removal_mask, result.image],
                              ["original", "removal mask", f"result · {result.timings['total']:.1f} s"]):
        ax.imshow(img if img.ndim == 2 else cv2.cvtColor(img, cv2.COLOR_BGR2RGB), cmap="gray")
        ax.set_title(title)
        ax.axis("off")
    fig.suptitle(f"“{instruction}”  [{result.route}{' · ' + result.insert_mode if result.insert_mode else ''}]")
    plt.tight_layout()
    plt.show()
    print("parsed :", result.parsed.to_dict())
    print("timings:", {k: round(v, 2) for k, v in result.timings.items()}, "| saved:", out_path)
    return result

## 3. Removal

Free-form phrasing, instance selection ("on the left") and a non-COCO object ("ladder", found by the
open-vocabulary fallback). The first call of each model also reports its load time as `load`.

In [ ]:
run("car_2.jpg", "Could you get rid of the dark sedan on the highway?")
run("park_2.jpg", "I don't want the man walking in this photo")
run("park.jpg", "remove the woman on the left")
run("multi.jpg", "delete the ladder")

## 4. Replacement with a text-defined object

The old object is erased with MI-GAN first; SD-1.5 inpainting (LCM-LoRA, 4 steps, TAESD) then draws the
new object inside a box-shaped placement mask on the clean background.

In [ ]:
run("car_2.jpg", "swap the car for a red bicycle")
run("cat.jpg", "turn the kitten into a golden retriever puppy")

## 5. Fast replacement (no diffusion)

Composites a stored cutout from `assets/` with colour matching and a contact shadow; ~3 s, but only for objects in the library.

In [ ]:
run("car_2.jpg", "replace the car with a bicycle", mode="composite")
run("park_2.jpg", "replace the person with a dog", mode="composite")

## 6. Try your own instruction

In [ ]:
IMAGE = "road_2.jpg"                     # any file in data/samples/, or edit run() to load your own
INSTRUCTION = "remove the bicycle"
run(IMAGE, INSTRUCTION)

## 7. What was loaded

In [ ]:
print("weights (MB):", json.dumps(editor.footprint(), indent=2))
print("load time (s):", {k: round(v, 1) for k, v in editor.load_seconds.items()})

Benchmarks on the full case set: `python scripts/run_benchmark.py --set final`.
Per-experiment notebooks with outputs: `notebooks/experiments/`.